# Draft-Revise Deep Validation

**Why**: Most robust CTM improvement (3/4 tasks positive, zero collapse).
sort +21pp, mazes +10pp, cifar10 +6pp in prior sweep.

**Goal**: 5 seeds x 4 tasks for statistical significance.

**Hardware**: 1 machine x 8 GPUs (~8h).

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

## Part A - Prior Results (st10)

How draft-revise performed in the initial sweep.

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    print(summary_stats(df_prior[df_prior.stage == 'st10']))
else:
    print('Prior data not found in csv_data/')

In [ ]:
if df_prior is not None:
    plot_prior_bar(df_prior, ['cifar10','mazes','parity','sort'],
                   'st10', 'revise', 'Prior: draft-revise vs baseline',
                   'figures/01_prior_bar.png')

In [ ]:
curves = load_prior_curves()
if curves:
    plot_prior_curves(curves, 'sort',
        [('st00','paper','baseline','#888'),
         ('st10','revise','revise','#9467bd')],
        'Sort convergence (prior)', 'figures/01_prior_conv.png')
    plot_prior_curves(curves, 'mazes',
        [('st00','paper','baseline','#888'),
         ('st10','revise','revise','#ff7f0e')],
        'Mazes convergence (prior)', 'figures/01_prior_conv_mazes.png')

## Part B - Experiment Design (5 seeds x 4 tasks = 20 runs)

Same config as st10 best: w=0.1, cp=0.15.

In [ ]:
exps = make_revise(['cifar10','mazes','sort','parity'], [0,1,2,3,4], w=0.1, cp=0.15)
print(f'{len(exps)} experiments')
for e in exps[:6]:
    print(f'  {e.name}')
print('  ...')

In [ ]:
# Dry run: preview commands
run_all(exps, gpus=8, log_root='logs/deep/01_revise', dry_run=True)

## Part C - Run Training

Set `CONFIRM_RUN = True` and re-run to launch. Takes ~8h on 8 GPUs.

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/01_revise')


In [ ]:
# Check progress (re-run anytime)
status('logs/deep/01_revise')

## Part D - Results Analysis

Run after training completes.

In [ ]:
df = collect('logs/deep/01_revise')
if df.empty:
    print('No results yet.')
else:
    print(df[['name','task','best_acc','delta']].to_string(index=False))
    plot_delta_bars(df, 'Draft-Revise vs baseline (5 seeds)', 'figures/01_delta.png')

In [ ]:
if not df.empty:
    plot_box_seeds(df, 'task', 'best_acc',
                   'Seed variance (5 seeds per task)', 'figures/01_box.png')

In [ ]:
if not df.empty:
    print(summary_stats(df, groupby=('task',)))